# Week 5 - Human Activity Recognition (HAR) using Deep Learning: Problem Definition

**Objective:** Implement HAR using classical machine learning *and* deep learning on the public **UCI HAR dataset**.

## a. Problem definition
HAR is a **classification** problem: motion-sensor data (accelerometer + gyroscope) from a smartphone is used to recognise what a person is doing.
Applications: health monitoring, sports analysis, smart-home automation, fall detection for the elderly.

**The data:** 30 volunteers (aged 19-48) wore a smartphone on the waist and performed 6 activities:

| ID | Activity |
|---|---|
| 1 | Walking |
| 2 | Walking upstairs |
| 3 | Walking downstairs |
| 4 | Sitting |
| 5 | Standing |
| 6 | Laying |

The phone recorded 3-axis linear acceleration and 3-axis angular velocity at 50 Hz. The raw signals were cut into **2.56-second windows** (128 readings, 50% overlap). From every window, **561 features** were engineered from the **time domain** (mean, std, min, max, energy ... of the signals) and the **frequency domain** (FFT-based features).

## Plan for the HAR experiment
| Week | Content |
|---|---|
| 5 | Problem definition (this notebook) + a first look at the files |
| 6 | Load the data, check quality, scale features |
| 7 | Univariate EDA |
| 8 | Classical ML models (LogReg, Random Forest, SVM) and deep models (1D-CNN, LSTM) |

## A first look at the dataset files
The dataset was downloaded from the UCI repository into `data/har_raw/UCI HAR Dataset/`. Let's see how it is organised.

In [1]:
import os
BASE = "data/har_raw/UCI HAR Dataset/"
for root, dirs, files in os.walk(BASE):
    if "Inertial Signals" in root:
        print(root.replace(BASE, ""), "->", len(files), "raw signal files (e.g.", files[0], ")")
        continue
    print(root.replace(BASE, "") or ".", "->", files)

. -> ['.DS_Store', 'activity_labels.txt', 'features.txt', 'features_info.txt', 'README.txt']
test -> ['subject_test.txt', 'X_test.txt', 'y_test.txt']
test\Inertial Signals -> 9 raw signal files (e.g. body_acc_x_test.txt )
train -> ['subject_train.txt', 'X_train.txt', 'y_train.txt']
train\Inertial Signals -> 9 raw signal files (e.g. body_acc_x_train.txt )


* `features.txt` - names of the 561 features
* `activity_labels.txt` - id -> activity name
* `train/` (7,352 windows from 21 volunteers) and `test/` (2,947 windows from 9 volunteers): `X_*.txt` = features, `y_*.txt` = activity id, `subject_*.txt` = volunteer id
* `Inertial Signals/` = the raw 128-sample sensor windows (not used by the record, which uses the 561 engineered features)

The train/test split is **by volunteer**, so the test set contains people the model has never seen - a realistic test.

In [2]:
import pandas as pd
activity_labels = pd.read_csv(BASE + "activity_labels.txt", sep=r"\s+", header=None, names=["id", "activity"])
features = pd.read_csv(BASE + "features.txt", sep=r"\s+", header=None, names=["idx", "feature"])
print(activity_labels.to_string(index=False))
print("\nTotal features:", len(features))
print("First 8 feature names:")
print(features["feature"].head(8).tolist())

subj_train = pd.read_csv(BASE + "train/subject_train.txt", header=None)[0]
subj_test = pd.read_csv(BASE + "test/subject_test.txt", header=None)[0]
print("\nVolunteers in train:", subj_train.nunique(), "| in test:", subj_test.nunique(), "| overlap:", len(set(subj_train) & set(subj_test)))

 id           activity
  1            WALKING
  2   WALKING_UPSTAIRS
  3 WALKING_DOWNSTAIRS
  4            SITTING
  5           STANDING
  6             LAYING

Total features: 561
First 8 feature names:
['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z', 'tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z', 'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y']

Volunteers in train: 21 | in test: 9 | overlap: 0


### Reading the feature names
Names such as `tBodyAcc-mean()-X` follow a pattern: **t** = time-domain, **f** = frequency-domain (after FFT); `BodyAcc` = body acceleration, `GravityAcc` = gravity component, `Gyro` = gyroscope; then the statistic (`mean()`, `std()`, `max()` ...) and the axis (X/Y/Z).

In [3]:
names = features["feature"]
print("Time-domain features     (start with 't'):", names.str.startswith("t").sum())
print("Frequency-domain features (start with 'f'):", names.str.startswith("f").sum())
print("Other (angle features)   :", (~names.str[0].isin(["t", "f"])).sum())

Time-domain features     (start with 't'): 265
Frequency-domain features (start with 'f'): 289
Other (angle features)   : 7


## Summary
* HAR = multi-class classification (6 classes) from smartphone motion sensors.
* Input: 561 engineered features per 2.56 s window. Output: one of 6 activities.
* Next week we load the data into DataFrames, check data quality and standardise the features.